In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Grad-CAM Figure\n",
        "ESC-50 / Hysper / US8K run folder?? checkpoint? ?? Grad-CAM figure? ?????."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "\n",
        "# Edit these\n",
        "RUN_DIR = Path(r\"H:/sound_classification_2/sound_classification_2/outputs/esc50_pooling_tradeoff_protocol/esc50_lwcnn_pool_AdaptSSRP_W4_Ks4812_h128_seed42\")\n",
        "FOLD = \"fold1\"\n",
        "SPLIT = \"test\"\n",
        "SAMPLE_INDEX = 0\n",
        "TARGET_CLASS = None   # None -> use predicted class\n",
        "TARGET_LAYER = None   # None -> auto pick final conv/relu block\n",
        "DEVICE = \"cuda\"\n",
        "ALPHA = 0.45\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import json\n",
        "import matplotlib.pyplot as plt\n",
        "import numpy as np\n",
        "import torch\n",
        "\n",
        "from experiments.esc50_pooling_tradeoff.data import build_dataset\n",
        "from experiments.esc50_pooling_tradeoff.gradcam_figure import (\n",
        "    GradCAM,\n",
        "    build_model,\n",
        "    choose_default_target_layer,\n",
        "    find_checkpoint,\n",
        "    get_module_by_name,\n",
        "    infer_num_classes_from_csv,\n",
        "    read_rows,\n",
        ")\n",
        "from experiments.esc50_pooling_tradeoff.train_eval_cv import resolve_device\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "fold_dir = RUN_DIR / FOLD\n",
        "ckpt_path = find_checkpoint(fold_dir)\n",
        "state = torch.load(ckpt_path, map_location=\"cpu\")\n",
        "cfg = state[\"config\"]\n",
        "data_cfg = dict(cfg[\"data\"])\n",
        "model_cfg = dict(cfg[\"model\"])\n",
        "device = torch.device(resolve_device(DEVICE))\n",
        "\n",
        "split_csv = fold_dir / \"normalized_csv\" / f\"{SPLIT}.csv\"\n",
        "rows = read_rows(split_csv)\n",
        "data_cfg[\"root\"] = str(Path.cwd())\n",
        "dataset = build_dataset(data_cfg, model_cfg, str(split_csv), training=False)\n",
        "x, y = dataset[SAMPLE_INDEX]\n",
        "x = x.unsqueeze(0).to(device)\n",
        "y_true = int(y.item())\n",
        "num_classes = int(state.get(\"num_classes\") or data_cfg.get(\"num_classes\") or infer_num_classes_from_csv(split_csv))\n",
        "model = build_model(cfg, num_classes=num_classes).to(device)\n",
        "model.load_state_dict(state[\"model_state\"], strict=True)\n",
        "model.eval()\n",
        "target_layer_name = TARGET_LAYER or choose_default_target_layer(model)\n",
        "target_layer = get_module_by_name(model, target_layer_name)\n",
        "print(\"checkpoint:\", ckpt_path)\n",
        "print(\"target_layer:\", target_layer_name)\n",
        "print(\"sample_row:\", rows[SAMPLE_INDEX])\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "cam_engine = GradCAM(model, target_layer)\n",
        "try:\n",
        "    with torch.enable_grad():\n",
        "        logits0 = model(x)\n",
        "        if isinstance(logits0, tuple):\n",
        "            logits0 = logits0[0]\n",
        "        pred_idx = int(logits0.argmax(dim=1).item())\n",
        "        class_idx = pred_idx if TARGET_CLASS is None else int(TARGET_CLASS)\n",
        "        logits, cam = cam_engine.compute(x, class_idx)\n",
        "finally:\n",
        "    cam_engine.remove()\n",
        "\n",
        "probs = torch.softmax(logits, dim=1).squeeze(0).detach().cpu()\n",
        "pred_prob = float(probs[pred_idx].item())\n",
        "mel = x.detach().cpu().squeeze(0).squeeze(0).numpy()\n",
        "cam_np = cam.detach().cpu().numpy()\n",
        "\n",
        "fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)\n",
        "axes[0].imshow(mel, origin=\"lower\", aspect=\"auto\", cmap=\"magma\")\n",
        "axes[0].set_title(\"Input log-mel\")\n",
        "axes[1].imshow(cam_np, origin=\"lower\", aspect=\"auto\", cmap=\"jet\")\n",
        "axes[1].set_title(\"Grad-CAM\")\n",
        "axes[2].imshow(mel, origin=\"lower\", aspect=\"auto\", cmap=\"gray\")\n",
        "axes[2].imshow(cam_np, origin=\"lower\", aspect=\"auto\", cmap=\"jet\", alpha=ALPHA)\n",
        "axes[2].set_title(\"Overlay\")\n",
        "for ax in axes:\n",
        "    ax.set_xlabel(\"Time\")\n",
        "    ax.set_ylabel(\"Mel bin\")\n",
        "fig.suptitle(f\"{RUN_DIR.name} | {FOLD} | pred={pred_idx} ({pred_prob:.3f}) | true={y_true} | target={class_idx}\", fontsize=10)\n",
        "plt.show()\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "out_dir = fold_dir / \"gradcam\"\n",
        "out_dir.mkdir(parents=True, exist_ok=True)\n",
        "stem = f\"{SPLIT}_idx{SAMPLE_INDEX:04d}_cls{pred_idx if TARGET_CLASS is None else int(TARGET_CLASS)}\"\n",
        "png_path = out_dir / f\"{stem}.png\"\n",
        "json_path = out_dir / f\"{stem}.json\"\n",
        "npy_path = out_dir / f\"{stem}_cam.npy\"\n",
        "fig.savefig(png_path, dpi=200, bbox_inches=\"tight\")\n",
        "np.save(npy_path, cam_np)\n",
        "payload = {\n",
        "    \"run_dir\": str(RUN_DIR),\n",
        "    \"fold\": FOLD,\n",
        "    \"checkpoint_path\": str(ckpt_path),\n",
        "    \"split\": SPLIT,\n",
        "    \"sample_index\": int(SAMPLE_INDEX),\n",
        "    \"sample_row\": rows[SAMPLE_INDEX],\n",
        "    \"true_class\": y_true,\n",
        "    \"pred_class\": pred_idx,\n",
        "    \"pred_prob\": pred_prob,\n",
        "    \"target_class\": pred_idx if TARGET_CLASS is None else int(TARGET_CLASS),\n",
        "    \"target_layer\": target_layer_name,\n",
        "    \"png_path\": str(png_path),\n",
        "    \"cam_npy_path\": str(npy_path),\n",
        "}\n",
        "json_path.write_text(json.dumps(payload, indent=2), encoding=\"utf-8\")\n",
        "print(\"saved png :\", png_path)\n",
        "print(\"saved json:\", json_path)\n",
        "print(\"saved npy :\", npy_path)\n"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.13"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}